In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# --- 统一绘图设置 ---
plt.rcParams.update({
    "axes.labelsize": 12,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7,
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), dpi=200)

# ==========================================
# 子图 1: Detail vs Realism (SD-1.5 相关)
# ==========================================
method_list1 = ["SD-1.5", "Diffusion-DPO", "Diffusion-KTO", "Ours"]
laplacian_raw = {"SD-1.5": 1832.336, "Diffusion-DPO": 1552.312, "Diffusion-KTO": 1580.512, "Ours": 2353.761}
sgp_raw = {"SD-1.5": -0.037 * 26, "Diffusion-DPO": -0.075 * 26, "Diffusion-KTO": -0.221 * 26, "Ours": -0.009 * 26}

def normalize(val, min_val, max_val):
    return (val - min_val) / (max_val - min_val)

laps = list(laplacian_raw.values()); sgps = list(sgp_raw.values())
lap_min, lap_max = min(laps), max(laps)
sgp_min, sgp_max = min(sgps), max(sgps)

style_map1 = {
    "SD-1.5":        {"mk": "o", "c": "#A4C6E2", "s": 160, "z": 3},
    "Diffusion-DPO": {"mk": "D", "c": "#AFD498", "s": 96, "z": 3},
    "Diffusion-KTO": {"mk": "P", "c": "#AE9DC7", "s": 120, "z": 3},
    "Ours":          {"mk": "s", "c": "#E4A39D", "s": 120, "z": 11}
}

for m in method_list1:
    x = normalize(sgp_raw[m], sgp_min, sgp_max)
    y = normalize(laplacian_raw[m], lap_min, lap_max)
    st = style_map1[m]
    ax1.scatter(x, y, s=st["s"], marker=st["mk"], facecolors=st["c"], edgecolors="black", linewidths=1.0, zorder=st["z"], label=m)

label_offsets1 = {"SD-1.5": (10, 0), "Diffusion-DPO": (10, -5), "Diffusion-KTO": (10, 0), "Ours": (10, -5)}
for m in method_list1:
    x = normalize(sgp_raw[m], sgp_min, sgp_max); y = normalize(laplacian_raw[m], lap_min, lap_max)
    dx, dy = label_offsets1.get(m, (5, 5))
    ax1.annotate(m, (x, y), xytext=(dx, dy), textcoords="offset points", fontsize=8, color="#333333", ha='left', va='center', zorder=20)

ax1.set_xlabel(r"Realism$\uparrow$")
ax1.set_ylabel(r"Detail$\uparrow$")
ax1.set_xlim(-0.1, 1.1); ax1.set_xticks(np.arange(0.0, 1.1, 0.2))
ax1.set_ylim(-0.1, 1.1); ax1.set_yticks(np.arange(0.0, 1.1, 0.2))
ax1.xaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.1f}"))
ax1.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.1f}"))
ax1.grid(True, linestyle="--", linewidth=0.8, alpha=0.3, zorder=0)
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
ax1.legend(loc="upper left", frameon=True, fancybox=True, edgecolor="gray", framealpha=0.3, borderpad=0.5)

# ==========================================
# 子图 2: Human Preference vs Stylization (SD-3.5 相关)
# ==========================================
plot_order2 = ["SD-3.5-M", "FlowGRPO", "GRPO-Guard", "DiffusionNFT", "Ours", "FlowGRPO+Ours"]
hps_dict = {
    "SD-3.5-M": { "UniRwd": 3.30, "HPSv3": 10.03 }, "FlowGRPO": { "UniRwd": 3.59, "HPSv3": 12.63 },
    "GRPO-Guard": { "UniRwd": 3.64, "HPSv3": 12.51 }, "DiffusionNFT": { "UniRwd": 3.68, "HPSv3": 12.47 },
    "Ours": { "UniRwd": 3.46, "HPSv3": 12.77 }, "FlowGRPO+Ours": { "UniRwd": 3.60, "HPSv3": 13.30 }
}
oneig_dict = {"SD-3.5-M": 0.357, "FlowGRPO": 0.260, "GRPO-Guard": 0.241, "DiffusionNFT": 0.175, "Ours": 0.305, "FlowGRPO+Ours": 0.273}

uni_vals = [hps_dict[m]["UniRwd"] for m in plot_order2]; hps_vals = [hps_dict[m]["HPSv3"] for m in plot_order2]
u_min, u_max = min(uni_vals), max(uni_vals); h_min, h_max = min(hps_vals), max(hps_vals)

style_map2 = {
    "SD-3.5-M": {"marker": "o", "color": "#A4C6E2"}, "FlowGRPO": {"marker": "D", "color": "#AFD498"}, 
    "GRPO-Guard": {"marker": "P", "color": "#AE9DC7"}, "DiffusionNFT": {"marker": "X", "color": "#e7dbd3"},
    "Ours": {"marker": "s", "color": "#E4A39D"}, "FlowGRPO+Ours": {"marker": "h", "color": "#D86983"}
}

for m in plot_order2:
    x = 0.5 * normalize(hps_dict[m]["UniRwd"], u_min, u_max) + 0.5 * normalize(hps_dict[m]["HPSv3"], h_min, h_max)
    y = oneig_dict[m]
    st = style_map2[m]
    s = 160 * 1.3 if m == "FlowGRPO+Ours" else (96 if m == "FlowGRPO" else (120 if m == "Ours" else 160))
    ax2.scatter(x, y, s=s, marker=st["marker"], facecolors=st["color"], edgecolors="black", linewidths=1.0, zorder=11 if m == "FlowGRPO+Ours" else 3, label=m)

label_offsets2 = {"SD-3.5-M": (10, 0), "FlowGRPO": (-8, 5), "GRPO-Guard": (10, -2), "DiffusionNFT": (-8, 5), "Ours": (-10, 0), "FlowGRPO+Ours": (1, 12)}
for m in plot_order2:
    x = 0.5 * normalize(hps_dict[m]["UniRwd"], u_min, u_max) + 0.5 * normalize(hps_dict[m]["HPSv3"], h_min, h_max)
    y = oneig_dict[m]
    dx, dy = label_offsets2.get(m, (5, 5))
    ha = 'right' if dx < 0 else ('left' if dx > 0 else 'center')
    ax2.annotate(m, (x, y), xytext=(dx, dy), textcoords="offset points", fontsize=8, color="#333333", ha=ha, va='center', zorder=20)

ax2.set_xlabel(r"Human Preference$\uparrow$")
ax2.set_ylabel(r"Stylization$\uparrow$")
ax2.set_xlim(-0.1, 1.1); ax2.set_xticks(np.arange(0.0, 1.1, 0.2))
ax2.set_ylim(0.14, 0.41); ax2.set_yticks(np.arange(0.15, 0.41, 0.05))
ax2.xaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.1f}"))
ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.2f}"))
ax2.grid(True, linestyle="--", linewidth=0.8, alpha=0.3, zorder=0)
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
ax2.legend(loc="lower left", frameon=True, fancybox=True, edgecolor="gray", framealpha=0.3, borderpad=0.5)

# 修正图例 Marker 大小 (通用逻辑)
for ax in [ax1, ax2]:
    leg = ax.get_legend()
    for handle in leg.legend_handles:
        lbl = handle.get_label()
        if "Ours" in lbl and "+" not in lbl: handle.set_sizes([60])
        elif "FlowGRPO" in lbl: handle.set_sizes([55])
        else: handle.set_sizes([80])

plt.tight_layout()
plt.savefig("combined_metrics.pdf", dpi=200, bbox_inches="tight")
plt.show()